# 7. Position Performance — QA

**Purpose** — Validate per-position PnL, fees, IL, time-in-range, and range reconstruction. Canonical home for TIR×PnL and single-position drill-down.

**Source tables**
- `py_position_performance_daily`
- `mart_nfpm_positions_current`
- `mart_nfpm_position_pnl_current`
- `mart_nfpm_position_lifecycle`
- `mart_nfpm_position_time_in_range_summary`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


**Window note**: `py_position_performance_daily` is currently populated for `2026-03-28 → 2026-04-03`. The notebook uses that window.

## §1 — Data fetch & coverage

In [3]:
PERF_START = "2026-03-28"
PERF_END   = "2026-04-03"

perf = db.read_query(f"""
SELECT
    token_id, day, price, tick,
    position_value, hodl_value, principal_wealth, total_wealth,
    il, fees_collected_value, fees_uncollected_est,
    pnl_net, pnl_vs_hodl,
    time_in_range_pct, share_of_pool, lvr_cumulative,
    is_in_range, tick_lower, tick_upper,
    reserve_x, reserve_y
FROM py_position_performance_daily
WHERE lower(pool) = lower('{POOL}')
  AND day >= toDate('{PERF_START}')
  AND day <= toDate('{PERF_END}')
ORDER BY token_id, day
""")
nfpm = db.read_query(f"""
SELECT token_id, current_owner, is_open, is_active, current_liquidity_raw,
       current_position_token0, mint_block_time
FROM mart_nfpm_positions_current
WHERE lower(pool) = lower('{POOL}')
""")
print(f"perf rows:   {perf.height} ({perf['day'].min()} → {perf['day'].max()})")
print(f"distinct token_ids in perf: {perf['token_id'].n_unique()}")
print(f"mart_nfpm_positions_current rows: {nfpm.height}")


perf rows:   796398 (2026-03-28 → 2026-04-03)
distinct token_ids in perf: 114249
mart_nfpm_positions_current rows: 116981


## §2 — Sanity checks

In [4]:
# Range validity
n_bad_range = perf.filter(pl.col("tick_lower") >= pl.col("tick_upper")).height
qa_check("tick_lower < tick_upper for all rows", n_bad_range == 0, detail=f"violations={n_bad_range}")

# Non-negativity
n_neg_pos = perf.filter(pl.col("position_value") < -1e-9).height
n_neg_fc  = perf.filter(pl.col("fees_collected_value") < -1e-9).height
n_neg_fu  = perf.filter(pl.col("fees_uncollected_est") < -1e-9).height
qa_check("position_value >= 0",       n_neg_pos == 0, detail=f"violations={n_neg_pos}")
qa_check("fees_collected_value >= 0", n_neg_fc == 0,  detail=f"violations={n_neg_fc}")
qa_check("fees_uncollected_est >= 0", n_neg_fu == 0,  detail=f"violations={n_neg_fu}")

# Finiteness
n_nan = perf.filter(pl.col("total_wealth").is_nan() | pl.col("total_wealth").is_null()).height
qa_check("total_wealth finite", n_nan == 0, detail=f"violations={n_nan}")

# TIR bounds
n_tir = perf.filter((pl.col("time_in_range_pct") < 0) | (pl.col("time_in_range_pct") > 1)).height
qa_check("time_in_range_pct ∈ [0, 1]", n_tir == 0, detail=f"violations={n_tir}")

# share_of_pool bounds
n_sp = perf.filter((pl.col("share_of_pool") < 0) | (pl.col("share_of_pool") > 1)).height
qa_check("share_of_pool ∈ [0, 1]", n_sp == 0, detail=f"violations={n_sp}")

🟢 PASS | tick_lower < tick_upper for all rows  —  violations=0
🟢 PASS | position_value >= 0  —  violations=0
🟢 PASS | fees_collected_value >= 0  —  violations=0
🟢 PASS | fees_uncollected_est >= 0  —  violations=0
🟢 PASS | total_wealth finite  —  violations=0
🟢 PASS | time_in_range_pct ∈ [0, 1]  —  violations=0
🟢 PASS | share_of_pool ∈ [0, 1]  —  violations=0


In [5]:
# Wealth reconciliation: total_wealth ≈ principal_wealth + fees_collected + fees_uncollected (within 1%)
sample = perf.filter(pl.col("day") == perf["day"].max()).filter(pl.col("total_wealth").abs() > 1.0).sample(n=min(50, perf.height), seed=42)
recon = sample.with_columns(
    (pl.col("principal_wealth") + pl.col("fees_collected_value") + pl.col("fees_uncollected_est")).alias("computed"),
).with_columns(
    ((pl.col("total_wealth") - pl.col("computed")).abs() / pl.col("total_wealth").abs().clip(1e-6, None)).alias("rel_resid"),
)
n_off = recon.filter(pl.col("rel_resid") > 0.01).height
qa_check("wealth reconciliation: total_wealth ≈ principal_wealth + fees (within 1%) on 50 sample positions",
         n_off / max(recon.height, 1) < 0.05, severity="warn",
         detail=f"off={n_off}/{recon.height}")


🟢 PASS | wealth reconciliation: total_wealth ≈ principal_wealth + fees (within 1%) on 50 sample positions  —  off=0/50


In [6]:
# Token coverage: count(distinct token_id) per day should be stable
cov = perf.group_by("day").agg(pl.col("token_id").n_unique().alias("n_tokens")).sort("day")
print(cov)
n_drop = cov.filter(
    pl.col("n_tokens") < (cov["n_tokens"].max() * 0.95)
).height
qa_check("daily token coverage ≥ 95% of max",
         n_drop == 0, severity="warn",
         detail=f"days_below_95pct={n_drop}")

shape: (7, 2)
┌────────────┬──────────┐
│ day        ┆ n_tokens │
│ ---        ┆ ---      │
│ date       ┆ u32      │
╞════════════╪══════════╡
│ 2026-03-28 ┆ 113379   │
│ 2026-03-29 ┆ 113472   │
│ 2026-03-30 ┆ 113574   │
│ 2026-03-31 ┆ 113675   │
│ 2026-04-01 ┆ 113847   │
│ 2026-04-02 ┆ 114202   │
│ 2026-04-03 ┆ 114249   │
└────────────┴──────────┘
🟢 PASS | daily token coverage ≥ 95% of max  —  days_below_95pct=0


## §3 — Diagnostic visuals

**QA purpose**: PnL distribution sanity — long tails are normal but extreme outliers (huge wins/losses) need cross-check.

In [7]:
terminal = perf.sort(["token_id", "day"]).group_by("token_id").tail(1)
import numpy as np
v = terminal["pnl_vs_hodl"].drop_nulls().to_numpy()
p1, p99 = np.percentile(v, [1, 99])
clipped = v[(v >= p1) & (v <= p99)]
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.08, subplot_titles=("PnL vs HODL distribution (1-99th pct)", "CDF (full range)"))
fig.add_trace(go.Histogram(x=clipped, nbinsx=80, marker_color="rgba(100,100,255,0.7)"), row=1, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="white", row=1, col=1)
sorted_pnl = sorted(v.tolist())
cum_pct = [i / len(sorted_pnl) for i in range(len(sorted_pnl))]
fig.add_trace(go.Scatter(x=sorted_pnl, y=cum_pct, mode="lines"), row=2, col=1)
fig.add_vline(x=0, line_dash="dash", line_color="white", row=2, col=1)
fig.update_yaxes(tickformat=".0%", title_text="% of positions", row=2, col=1)
fig.update_layout(height=560, title_text="PnL distribution (terminal day)", showlegend=False)
fig.show()
print(f"positions: {terminal.height}  |  win-rate vs HODL: {(terminal['pnl_vs_hodl'] > 0).mean():.1%}")

positions: 114249  |  win-rate vs HODL: 54.9%


**QA purpose**: fees vs IL by range bucket — validates the trade-off shape across width buckets.

In [8]:
import numpy as np
range_width = (terminal["tick_upper"].cast(pl.Int64) - terminal["tick_lower"].cast(pl.Int64))
bucketed = terminal.with_columns(
    range_width.alias("range_width"),
    pl.when(range_width < 200).then(pl.lit("1. <200"))
    .when(range_width < 1000).then(pl.lit("2. 200-1k"))
    .when(range_width < 5000).then(pl.lit("3. 1k-5k"))
    .otherwise(pl.lit("4. >5k")).alias("bucket"),
)
agg = bucketed.group_by("bucket").agg(
    pl.len().alias("n"),
    pl.col("fees_collected_value").mean().alias("avg_fees"),
    pl.col("il").mean().alias("avg_il"),
    pl.col("pnl_vs_hodl").mean().alias("avg_pnl"),
    pl.col("time_in_range_pct").mean().alias("avg_tir"),
).sort("bucket")
fig = go.Figure()
fig.add_trace(go.Bar(x=agg["bucket"].to_list(), y=agg["avg_fees"].to_list(), name="avg fees",  marker_color="#22c55e"))
fig.add_trace(go.Bar(x=agg["bucket"].to_list(), y=agg["avg_il"].to_list(),    name="avg IL",     marker_color="#ef4444"))
fig.add_trace(go.Scatter(x=agg["bucket"].to_list(), y=agg["avg_pnl"].to_list(), name="avg net PnL", mode="lines+markers", line=dict(color="white", width=2)))
fig.update_layout(height=380, barmode="group", title_text="Fees vs IL by range bucket")
fig.show()
print(agg)

shape: (4, 6)
┌───────────┬───────┬─────────────┬─────────────┬─────────────┬─────────┐
│ bucket    ┆ n     ┆ avg_fees    ┆ avg_il      ┆ avg_pnl     ┆ avg_tir │
│ ---       ┆ ---   ┆ ---         ┆ ---         ┆ ---         ┆ ---     │
│ str       ┆ u32   ┆ f64         ┆ f64         ┆ f64         ┆ f64     │
╞═══════════╪═══════╪═════════════╪═════════════╪═════════════╪═════════╡
│ 1. <200   ┆ 46067 ┆ 525.138914  ┆ -344.650413 ┆ 180.634259  ┆ 0.0     │
│ 2. 200-1k ┆ 23987 ┆ 2216.928106 ┆ -238.084824 ┆ 1978.937839 ┆ 0.0     │
│ 3. 1k-5k  ┆ 32580 ┆ 4245.358882 ┆ -596.41941  ┆ 3648.971506 ┆ 0.0     │
│ 4. >5k    ┆ 11615 ┆ 4483.162927 ┆ -284.504072 ┆ 4198.681292 ┆ 0.0     │
└───────────┴───────┴─────────────┴─────────────┴─────────────┴─────────┘


**QA purpose**: time-in-range bimodality — should peak at 0 and 1 (closed positions / always-active).

In [9]:
fig = go.Figure(go.Histogram(x=terminal["time_in_range_pct"].to_list(), nbinsx=20))
fig.update_layout(height=300, title_text="Time-in-range distribution")
fig.show()

**QA purpose**: TIR vs PnL scatter — confirms TIR is positively (but noisily) correlated with PnL.

In [10]:
fee_il = terminal.filter(pl.col("total_wealth").abs() > 0.01)
fig = go.Figure(go.Scatter(
    x=fee_il["time_in_range_pct"].to_list(),
    y=fee_il["pnl_vs_hodl"].to_list(),
    mode="markers",
    marker=dict(size=4, color=fee_il["fees_collected_value"].to_list(), colorscale="Viridis", colorbar_title="fees", opacity=0.4),
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(height=420, title_text="Time-in-range vs PnL (color = fees)", xaxis_tickformat=".0%", xaxis_title="TIR", yaxis_title="PnL vs HODL")
fig.show()

**QA purpose**: range map — top 50 positions vs current tick. Validates range reconstruction visually.

In [11]:
top50 = terminal.sort("total_wealth", descending=True).head(50).sort("tick_lower")
current_tick = int(terminal["tick"].mode().to_list()[0])
fig = go.Figure()
import math
for i, row in enumerate(top50.to_dicts()):
    tl, tu = int(row["tick_lower"]), int(row["tick_upper"])
    pnl = float(row["pnl_vs_hodl"])
    width = max(2, min(12, math.log1p(abs(float(row["total_wealth"]))) * 1.5))
    color = "rgba(0,200,0,0.7)" if pnl > 0 else "rgba(200,0,0,0.7)"
    fig.add_trace(go.Scatter(x=[tl, tu], y=[i, i], mode="lines", line=dict(width=width, color=color), showlegend=False,
                              hovertemplate=f"token={row['token_id']}<br>range=[{tl}, {tu}]<br>PnL={pnl:.2f}<extra></extra>"))
fig.add_vline(x=current_tick, line_dash="dash", line_color="yellow", annotation_text=f"current tick ({current_tick})")
fig.update_layout(height=520, title_text="Range map — top 50 (green=profitable, width=wealth)", yaxis=dict(showticklabels=False))
fig.show()

**QA purpose**: PnL reconciliation table for 5 selected positions — initial / current / fees / hodl / residual.

In [12]:
sample5 = terminal.sort("total_wealth", descending=True).head(5)
recon_rows = []
for r in sample5.to_dicts():
    computed = r["principal_wealth"] + r["fees_collected_value"] + r["fees_uncollected_est"]
    recon_rows.append({
        "token_id": r["token_id"],
        "principal_wealth": r["principal_wealth"],
        "position_value": r["position_value"],
        "fees_collected": r["fees_collected_value"],
        "fees_uncollected": r["fees_uncollected_est"],
        "computed_total": computed,
        "total_wealth": r["total_wealth"],
        "residual": r["total_wealth"] - computed,
        "hodl_value": r["hodl_value"],
        "pnl_vs_hodl": r["pnl_vs_hodl"],
    })
recon_df = pl.DataFrame(recon_rows)
recon_df


token_id,principal_wealth,position_value,fees_collected,fees_uncollected,computed_total,total_wealth,residual,hodl_value,pnl_vs_hodl
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
253158,1.3578e8,0.0,534551.228157,0.0,1.3631e8,1.3631e8,0.0,1.3813e8,-1.8173e6
921260,1.3362e8,0.0,263781.090822,0.0,1.3388e8,1.3388e8,0.0,1.3362e8,263781.09082
923861,1.3226e8,2.5890e-9,284162.358846,0.0,1.3254e8,1.3254e8,0.0,1.3217e8,370042.818878
733923,1.0593e8,0.0,281541.940427,0.0,1.0621e8,1.0621e8,0.0,1.0593e8,281541.940425
190489,9.3055e7,0.0,1.5792e6,0.0,9.4634e7,9.4634e7,0.0,1.0101e8,-6.3750e6


**QA purpose**: top 5 deep-dive — wealth/HODL/fees/IL trajectories per position. Visual sanity per series.

In [13]:
top_ids = terminal.sort("total_wealth", descending=True).head(5)["token_id"].to_list()
for tid in top_ids:
    pos = perf.filter(pl.col("token_id") == tid).sort("day")
    if pos.height < 2:
        continue
    days = pos["day"].to_list()
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=("Wealth vs HODL", "Fees vs IL"))
    fig.add_trace(go.Scatter(x=days, y=pos["total_wealth"].to_list(), name="Wealth"),     row=1, col=1)
    fig.add_trace(go.Scatter(x=days, y=pos["hodl_value"].to_list(),   name="HODL", line=dict(dash="dot")), row=1, col=1)
    fig.add_trace(go.Scatter(x=days, y=pos["fees_collected_value"].to_list(), name="Fees", line=dict(color="#22c55e")), row=2, col=1)
    fig.add_trace(go.Scatter(x=days, y=pos["il"].to_list(),                    name="IL",   line=dict(color="#ef4444")), row=2, col=1)
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=2, col=1)
    fig.update_layout(height=420, title_text=f"Token {tid} — wealth, fees, IL")
    fig.show()

## §4 — QA verdict

In [14]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: PASS
  PASS: 9   WARN: 0   FAIL: 0


label,status,severity,detail
str,str,str,str
"""tick_lower < tick_upper for al…","""PASS""","""fail""","""violations=0"""
"""position_value >= 0""","""PASS""","""fail""","""violations=0"""
"""fees_collected_value >= 0""","""PASS""","""fail""","""violations=0"""
"""fees_uncollected_est >= 0""","""PASS""","""fail""","""violations=0"""
"""total_wealth finite""","""PASS""","""fail""","""violations=0"""
"""time_in_range_pct ∈ [0, 1]""","""PASS""","""fail""","""violations=0"""
"""share_of_pool ∈ [0, 1]""","""PASS""","""fail""","""violations=0"""
"""wealth reconciliation: total_w…","""PASS""","""warn""","""off=0/50"""
"""daily token coverage ≥ 95% of …","""PASS""","""warn""","""days_below_95pct=0"""
